# TrOCR-small Smoke Test & LoRA Layer Verification

**Requirements**:
1. Base model: `microsoft/trocr-small-handwritten` (62M).
2. LoRA recipe: rank 8 on query and value projections of the last two visual-encoder attention blocks (`layer.10` and `layer.11`).
3. Decoder strictly frozen.

In [ ]:
# Step 1: Install dependencies
!pip install -q "sentencepiece>=0.2.0" "protobuf>=3.20.0" "transformers>=4.46.0" "peft>=0.13.0" pillow torch torchvision

In [ ]:
# Step 2: Self-healing runtime check
# If sentencepiece was just installed, this restarts the kernel once to reload C-extensions.
# Subsequent runs will pass through with [OK].
import os
try:
    import sentencepiece
    print(f"[OK] Environment ready: sentencepiece {sentencepiece.__version__} is loaded.")
except ImportError:
    print("sentencepiece not yet loaded in Python kernel. Restarting kernel...")
    os.kill(os.getpid(), 9)

In [ ]:
# Step 3: Load TrOCR-small processor and model
import torch
from transformers import AutoImageProcessor, XLMRobertaTokenizer, TrOCRProcessor, VisionEncoderDecoderModel

MODEL_ID = "microsoft/trocr-small-handwritten"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Direct initialization bypasses AutoTokenizer's TokenizersBackend bug on vision-encoder-decoder models
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_ID)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
model.config.decoder_start_token_id = processor.tokenizer.eos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print(f"[SUCCESS] Loaded {MODEL_ID}")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Tokenizer class: {type(processor.tokenizer).__name__}")

In [ ]:
# Step 4: List all named query/value attention modules in the encoder
import re

print("All encoder attention query/value modules:")
found = 0
for name, module in model.named_modules():
    if not name.startswith("encoder."):
        continue
    if any(name.endswith(k) for k in [".q_proj", ".v_proj", ".query", ".value"]):
        match = re.search(r"layer[s]?\.(\d+)", name)
        block_str = f"Block {int(match.group(1)):02d}" if match else "Unknown"
        print(f"  [{block_str}] {name}")
        found += 1

print(f"\nTotal query/value modules found: {found}")

In [ ]:
# Step 5: Apply LoRA (rank 8) to the last two encoder attention blocks' query+value projections only
import re
from peft import LoraConfig, get_peft_model

# Bypass pre-installed Colab torchao 0.10.0 version check in PEFT
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

# 1. Dynamically discover all encoder query/value attention modules directly from model
encoder_attn = []
for name, module in model.named_modules():
    if not name.startswith("encoder."):
        continue
    if any(name.endswith(k) for k in [".q_proj", ".v_proj", ".query", ".value"]):
        match = re.search(r"layer[s]?\.(\d+)", name)
        if match:
            encoder_attn.append((int(match.group(1)), name))

encoder_attn = sorted(encoder_attn, key=lambda x: (x[0], x[1]))

if not encoder_attn:
    raise RuntimeError(
        "Could not detect encoder query/value modules. "
        f"Available modules with attention:\n{[n for n, _ in model.named_modules() if 'attention' in n][:10]}"
    )

# 2. Select the last two encoder blocks
max_layer = max(idx for idx, _ in encoder_attn)
last_two_layers = (max_layer - 1, max_layer)
target_modules = [name for idx, name in encoder_attn if idx in last_two_layers]

print(f"Detected {len(encoder_attn)} encoder attention projections across {max_layer + 1} blocks.")
print(f"Targeting last two blocks ({last_two_layers[0]} & {last_two_layers[1]}):\n")
for t in target_modules:
    print("  ", t)

assert len(target_modules) == 4, f"Expected exactly 4 target modules, got {len(target_modules)}: {target_modules}"

# 3. Create LoRA adapter (rank 8, decoder frozen)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=target_modules,
    bias="none",
)

peft_model = get_peft_model(model, lora_config)

# 4. Verify trainable parameters and confirm decoder is strictly frozen
print("\n--- Verification Results ---")
peft_model.print_trainable_parameters()

trainable_params = [name for name, p in peft_model.named_parameters() if p.requires_grad]
decoder_trainable = [p for p in trainable_params if "decoder" in p]

print(f"\nDecoder trainable parameters: {len(decoder_trainable)} (Must be 0)")
assert len(decoder_trainable) == 0, "ERROR: Decoder parameters are not frozen."

print(f"\nTrainable parameter tensors ({len(trainable_params)} total):\n")
for p in trainable_params:
    print("  ", p)

print("\n[OK] Smoke test passed: TrOCR-small loaded, LoRA targets confirmed, decoder strictly frozen.")